# Intro

In this notebook you will see how I cleaned data from this [dataset](https://www.kaggle.com/rikdifos/credit-card-approval-prediction) on credit cards to create a clean, easier to use version. You can find the resulting dataset [here](https://www.kaggle.com/samuelcortinhas/credit-card-classification-clean-data).

# Libraries

In [ ]:
# Core
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set(style='darkgrid', font_scale=1.4)
from imblearn.over_sampling import SMOTE
import itertools

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score, precision_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.feature_selection import mutual_info_classif

# Models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier

# Data

In [ ]:
# Save to df
applications = pd.read_csv("../input/credit-card-approval-prediction/application_record.csv", encoding = 'utf-8')
record = pd.read_csv("../input/credit-card-approval-prediction/credit_record.csv", encoding = 'utf-8')

# Shape and preview
print('Application record df shape:', applications.shape)
applications.head()

In [ ]:
# Shape and preview
print('Credit record df shape:', record.shape)
record.head()

**Missing values**

In [ ]:
applications.isna().sum()

In [ ]:
record.isna().sum()

**Duplicates**

In [ ]:
print(f'Duplicates in applications df: {applications.iloc[:,1:].duplicated().sum()}, ({np.round(100*applications.iloc[:,1:].duplicated().sum()/len(applications),1)}%)')
print('')
print(f'Duplicates in record df: {record.duplicated().sum()}, ({np.round(100*record.duplicated().sum()/len(record),1)}%)')

**Cardinality of features**

There is 1 constant feature, 6 binary features, 3 continuous features and 7 non-binary categorical features.

In [ ]:
applications.nunique()

# Data Cleaning

**Drop duplicates**

In [ ]:
# Drop duplicates
applications=applications.drop_duplicates(subset=applications.columns[1:], keep='first')

**Fill missing values**

In [ ]:
# Fill missing values
applications['OCCUPATION_TYPE'].fillna(value='Other', inplace=True)

**Drop constant feature**

This won't be useful to our models.

In [ ]:
# Drop constant feature
applications.drop('FLAG_MOBIL', axis=1, inplace=True)

**Create target variable**

Because the credit records are incomplete (the ID ranges in both dfs don't match) we have to create our own target. We do this by identifying 'low risk' and 'high risk' credit users. We say a user is 'high risk' if during any month they are late on payments by 30 days or more, and 'low risk' otherwise. This is the target we will try to predict later.

In [ ]:
# Construct target
record['target']=record['STATUS']
record['target'].replace('X', 0, inplace=True)
record['target'].replace('C', 0, inplace=True)
record['target']=record['target'].astype(int)
record.loc[record['target']>=1,'target']=1

# target=1 (high risk) iff there is at least one month where user is late on payments by 30 days or more
target_df=pd.DataFrame(record.groupby(['ID'])['target'].agg(max)).reset_index()

# Merge dataframes
new_df=pd.merge(applications, target_df, how='inner', on=['ID'])

**Account length**

It makes sense that the number of months the account has been open for correlates with the riskiness of the customer (since there are more opportunity to miss payments).

In [ ]:
# Extract how many months account has been open for
start_df=pd.DataFrame(record.groupby(['ID'])['MONTHS_BALANCE'].agg(min)).reset_index()

# Rename column
start_df.rename(columns={'MONTHS_BALANCE':'ACCOUNT_LENGTH'}, inplace=True)

# Make entries positive
start_df['ACCOUNT_LENGTH']=-start_df['ACCOUNT_LENGTH']

# Merge dataframes on ID
new_df=pd.merge(new_df, start_df, how='inner', on=['ID'])

**Continuous features**

In [ ]:
# Create age feature
new_df['AGE_YEARS']=-new_df['DAYS_BIRTH']/365.2425
new_df.drop('DAYS_BIRTH', axis=1, inplace=True)

# Create unemployed indicator
new_df['UNEMPLOYED']=0
new_df.loc[-new_df['DAYS_EMPLOYED']<0,'UNEMPLOYED']=1

# Create years employed feature
new_df['YEARS_EMPLOYED']=-new_df['DAYS_EMPLOYED']/365.2425
new_df.loc[new_df['YEARS_EMPLOYED']<0,'YEARS_EMPLOYED']=0
new_df.drop('DAYS_EMPLOYED', axis=1, inplace=True)

**Encoding categorical features**

In [ ]:
# Encode binary features
new_df["CODE_GENDER"] =  new_df["CODE_GENDER"].replace(['F','M'],[0,1])
new_df["FLAG_OWN_CAR"] = new_df["FLAG_OWN_CAR"].replace(["Y","N"],[1,0])
new_df["FLAG_OWN_REALTY"] = new_df["FLAG_OWN_REALTY"].replace(["Y","N"],[1,0])

**Rename columns**

In [ ]:
new_df = new_df.rename(columns={'CODE_GENDER': 'Gender', 'FLAG_OWN_CAR': 'Own_car', 
                               'FLAG_OWN_REALTY':'Own_property','CNT_CHILDREN':'Num_children',
                               'AMT_INCOME_TOTAL':'Total_income','NAME_INCOME_TYPE':'Income_type',
                               'NAME_EDUCATION_TYPE':'Education_type','NAME_FAMILY_STATUS':'Family_status',
                               'NAME_HOUSING_TYPE':'Housing_type','FLAG_WORK_PHONE':'Work_phone',
                               'FLAG_PHONE':'Phone','FLAG_EMAIL':'Email','OCCUPATION_TYPE':'Occupation_type',
                               'CNT_FAM_MEMBERS':'Num_family','target':'Target','ACCOUNT_LENGTH':'Account_length',
                                'AGE_YEARS':'Age','UNEMPLOYED':'Unemployed','YEARS_EMPLOYED':'Years_employed'})

**Data type**

In [ ]:
new_df['Num_family']=new_df['Num_family'].astype(int)

**Re-order columns**

In [ ]:
new_df.columns

In [ ]:
new_df=new_df[['ID', 'Gender', 'Own_car', 'Own_property', 'Work_phone',
               'Phone', 'Email', 'Unemployed', 'Num_children', 'Num_family', 
               'Account_length', 'Total_income', 'Age', 'Years_employed',  
               'Income_type', 'Education_type', 'Family_status',
               'Housing_type', 'Occupation_type','Target']]

# Output

In [ ]:
# Shape and preview
print('New df shape:', new_df.shape)
new_df.head()

In [ ]:
# Data types
new_df.dtypes

In [ ]:
# Save to csv
new_df.to_csv('clean_data.csv',index=False)